# Universidad del Istmo
# Tarea 4 — Introducción a Series de Tiempo
**Curso:** Business Intelligence  
**Estudiante:** Ximena Valenzuela  
**Fecha de entrega:** 25 de octubre de 2026

**Ejecución:** `Online Retail.xlsx` y `limpieza.py` deben estar junto a este notebook. Ejecutar todas las celdas en orden. Los cálculos SARIMA con origen móvil tardan varios minutos.

> **Estado:** notebook de trabajo completo a nivel de implementación, pendiente de ejecutar íntegramente y revisar interpretaciones de los resultados. No se deben presentar valores no calculados como conclusiones definitivas.

## Parte 0 — Procesos simulados (0.1 a 0.5)
Incluye demostraciones, simulación AR(1), pruebas de autocorrelación y Ljung–Box.

## 0.1 Derivaciones a mano
Sea $Y_t=c+\phi Y_{t-1}+\varepsilon_t$, con $E[\varepsilon_t]=0$, $Var(\varepsilon_t)=\sigma^2$, errores independientes del pasado y $|\phi|<1$.

**Media.** Por estacionariedad $E[Y_t]=E[Y_{t-1}]=\mu$:
$$\mu=c+\phi\mu\implies \boxed{\mu=\frac{c}{1-\phi}}.$$

**Varianza.** Como el error es independiente de $Y_{t-1}$:
$$Var(Y_t)=\phi^2 Var(Y_{t-1})+\sigma^2.$$
Por estacionariedad $\gamma_0=\phi^2\gamma_0+\sigma^2$, así que
$$\boxed{Var(Y_t)=\frac{\sigma^2}{1-\phi^2}}.$$

**Autocorrelación.** Al restar $\mu$, $Y_t-\mu=\phi(Y_{t-1}-\mu)+\varepsilon_t$. Multiplicando por $(Y_{t-k}-\mu)$ y tomando esperanza, para $k\geq1$:
$$\gamma_k=\phi\gamma_{k-1}\implies \gamma_k=\phi^k\gamma_0\implies\boxed{\rho_k=\phi^k}.$$

**Caminata aleatoria.** Con $\phi=1,c=0,Y_0=0$, $Y_t=\sum_{j=1}^t\varepsilon_j$ y, por independencia,
$$\boxed{Var(Y_t)=t\sigma^2}.$$
No es débilmente estacionaria porque la varianza aumenta con $t$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2
from statsmodels.stats.diagnostic import acorr_ljungbox

N = 300
R = 5000
PHIS = [0, 0.5, 0.9, 1.0]

In [ ]:
def acf_manual(y, k):
    y = np.asarray(y, dtype=float)
    z = y - y.mean()
    return np.dot(z[k:], z[:-k]) / np.dot(z, z)

def ols_phi(y):
    x, z = y[:-1], y[1:]
    return np.dot(x - x.mean(), z - z.mean()) / np.dot(x - x.mean(), x - x.mean())

def simular_ar1(phi, rng, n=N, burn=200):
    eps = rng.normal(size=n + burn)
    y = np.zeros(n + burn)
    for t in range(1, n + burn):
        y[t] = phi * y[t-1] + eps[t]
    return y[burn:]

def resumen(valores):
    a = np.asarray(valores)
    return a.mean(), a.std(ddof=1)/np.sqrt(len(a))

## 0.2 Simulación: realizaciones y 5,000 réplicas por escenario

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 7))
for ax, phi, seed in zip(axs.flat, PHIS, [2026, 2027, 2028, 2029]):
    y = simular_ar1(phi, np.random.default_rng(seed))
    ax.plot(y, lw=1)
    ax.set_title(f"AR(1), phi={phi}")
    ax.set_xlabel("t")
plt.tight_layout()
plt.show()

In [ ]:
filas=[]
for phi, seed in zip(PHIS, [101, 102, 103, 104]):
    rng=np.random.default_rng(seed)
    r1=[];r5=[];rechazo=[];estimaciones=[]
    for _ in range(R):
        y=simular_ar1(phi,rng)
        a=acf_manual(y,1); bb=acf_manual(y,5)
        r1.append(a);r5.append(bb)
        rechazo.append(abs(a)>1.96/np.sqrt(N))
        estimaciones.append(ols_phi(y))
    m1,s1=resumen(r1);m5,s5=resumen(r5)
    mr,sr=resumen(rechazo);mp,sp=resumen(estimaciones)
    filas.append({'phi':phi,'r1 medio':m1,'EE r1':s1,'r5 medio':m5,'EE r5':s5,
                  'phi^5 teórico':phi**5,'rechazo r1':mr,'EE rechazo':sr,
                  'phi OLS medio':mp,'EE phi OLS':sp})
tabla_ar=pd.DataFrame(filas)
display(tabla_ar.round(5))

## 0.3 Interpretación (completar con la tabla obtenida)
- Para $\phi=0$, la tasa de rechazo esperada es aproximadamente 5%, porque la prueba tiene nivel de significancia 0.05.
- Para $\phi=1$, la fórmula $ho_5=\phi^5$ **no aplica**: exige estacionariedad y la caminata aleatoria no la cumple.
- Comparar la estimación media OLS con el $\phi$ verdadero: en muestras finitas puede haber sesgo; documentar dirección y magnitud observadas.
- Descartar 200 pasos no estacionariza una caminata aleatoria, pero la ACF muestral resta la media y no cambia ante un desplazamiento constante de toda la trayectoria.

## 0.4 Tendencia y patrón semanal sin memoria

In [ ]:
def simular_media(tipo, rng, n=N):
    t=np.arange(1,n+1)
    if tipo=='tendencia': m=2*t/n
    elif tipo=='semanal': m=np.where(t%7==6,-1.5,0.25)
    elif tipo=='blanco': m=np.zeros(n)
    else: raise ValueError(tipo)
    return m+rng.normal(size=n)

filas=[]
for tipo,seed in [('blanco',300),('tendencia',301),('semanal',302)]:
    rng=np.random.default_rng(seed)
    resultados=[]
    for _ in range(R):
        y=simular_media(tipo,rng)
        a=acf_manual(y,1)
        resultados.append([a,acf_manual(y,5),acf_manual(y,7),
                           float(abs(a)>1.96/np.sqrt(N))])
    arr=np.array(resultados)
    fila={'proceso':tipo}
    for j,nombre in enumerate(['r1','r5','r7','rechazo r1']):
        fila[nombre]=arr[:,j].mean()
        fila['EE '+nombre]=arr[:,j].std(ddof=1)/np.sqrt(R)
    filas.append(fila)
display(pd.DataFrame(filas).round(5))

## 0.5 Ljung–Box (m=10)

In [ ]:
def ljung_box_manual(y,m=10):
    n=len(y)
    r=np.array([acf_manual(y,k) for k in range(1,m+1)])
    Q=n*(n+2)*np.sum(r**2/(n-np.arange(1,m+1)))
    return Q,chi2.sf(Q,m)

filas=[]
for tipo,seed in [('blanco',401),('tendencia',402),('semanal',403)]:
    rng=np.random.default_rng(seed)
    rechazos=[]
    for _ in range(R):
        y=simular_media(tipo,rng)
        Q,p=ljung_box_manual(y,10)
        rechazos.append(p<0.05)
    media,ee=resumen(rechazos)
    filas.append({'proceso':tipo,'tasa rechazo LB(10)':media,'EE':ee})
display(pd.DataFrame(filas).round(5))

# Verificación independiente en una muestra
muestra=simular_media('blanco',np.random.default_rng(999))
print('Manual:',ljung_box_manual(muestra,10))
print('statsmodels:')
display(acorr_ljungbox(muestra,lags=[10],return_df=True))

## Interpretación final de 0.4 y 0.5 (redactar tras ejecutar)
La tendencia puede generar autocorrelación muestral sin dependencia en los errores, debido a una media cambiante. El patrón semanal puede mostrar estructura especialmente en el rezago 7, aun si la prueba de rezago 1 no lo detecta. Ljung–Box combina información de varios rezagos y puede identificar esa estructura. La comparación con RANDU debe enfocarse en que una prueba aislada no garantiza ausencia de estructura: hay patrones que se descubren mediante diagnósticos adicionales. Verificar estas conclusiones contra los resultados calculados.

## Parte 1 — Limpieza de Online Retail (1.1 a 1.5)
Se examinan anomalías, se justifica la limpieza, se crea la bitácora y se revisan las seis transacciones extremas.

### Cargar los datos como texto para no perder códigos o ceros iniciales

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from limpieza import preparar, limpiar, CODIGOS_ADMIN

ruta = Path("Online Retail.xlsx")
if not ruta.exists():
    raise FileNotFoundError(f"No encuentro {ruta.resolve()}. Colocá el Excel en la misma carpeta del notebook.")
df = pd.read_excel(ruta, engine="openpyxl",
                   dtype={"InvoiceNo":"string","StockCode":"string","CustomerID":"string"})
display(df.head(10))

## 1.1 Tamaño de la base y faltantes

In [ ]:
print("Total de filas:", len(df))
print("Total de facturas:", df["InvoiceNo"].nunique())
print("Total de productos:", df["StockCode"].nunique())
print("Total de clientes:", df["CustomerID"].nunique())
print("Fracción sin CustomerID:", round(df["CustomerID"].isna().mean(), 4))
print("Porcentaje sin CustomerID:", round(df["CustomerID"].isna().mean()*100, 2), "%")
df.info()

**Resultados verificados con `Online Retail.xlsx`:** 541,909 filas; 25,900 facturas únicas; 4,070 códigos de producto; 4,372 clientes identificados; 135,080 filas sin `CustomerID` (24.93 %). Para el análisis de ventas, conservar registros sin cliente; para RFM, trabajar solo con identificados.

## 1.2 Investigación antes de decidir qué eliminar

In [ ]:
x = preparar(df)
x["es_C"] = x["InvoiceNo"].str.upper().str.match(r"^C[0-9]", na=False)
x["factura_no_numerica"] = ~x["InvoiceNo"].str.match(r"^[0-9]", na=False)
x["stock_no_5digitos"] = ~x["StockCode"].str.match(r"^[0-9]{5}", na=False)
x["duplicado"] = pd.util.hash_pandas_object(x.drop(columns=["importe","es_C","factura_no_numerica","stock_no_5digitos"], errors="ignore"), index=False).duplicated()
print("Duplicados exactos:", int(x["duplicado"].sum()))
display(x.loc[x["duplicado"]].head(12))

In [ ]:
print("Prefijos de facturas que no empiezan con dígito:")
display(x.loc[x["factura_no_numerica"]]
        .assign(prefijo=lambda z:z["InvoiceNo"].str.extract(r"^([^0-9]+)", expand=False))
        .groupby("prefijo", dropna=False)
        .agg(filas=("InvoiceNo","size"), importe=("importe","sum"))
        .sort_values("filas", ascending=False))
print("Cancelaciones C: conservamos sus importes negativos si son válidos.")
display(x.loc[x["es_C"], ["InvoiceNo","Quantity","UnitPrice","importe","Description"]].head(10))

In [ ]:
print("Códigos que no empiezan con cinco dígitos: investigar por descripción")
codigos = (x.loc[x["stock_no_5digitos"]]
           .groupby(["StockCode","Description"],dropna=False)
           .agg(filas=("StockCode","size"), importe=("importe","sum"))
           .sort_values("filas",ascending=False))
display(codigos.head(70))
print("Códigos administrativos propuestos:", sorted(CODIGOS_ADMIN))
print("IMPORTANTE: no todos los códigos alfanuméricos son cargos; revisar productos reales.")

In [ ]:
print("Precios cero o negativos:")
display(x.loc[x["UnitPrice"] <= 0,
              ["InvoiceNo","StockCode","Description","Quantity","UnitPrice","importe"]].head(35))
print("Cantidad negativa SIN factura C:")
display(x.loc[(x["Quantity"] < 0) & ~x["es_C"],
              ["InvoiceNo","StockCode","Description","Quantity","UnitPrice","importe"]].head(35))
print("Sin CustomerID:",x["CustomerID"].isna().sum())
print("Importe sin CustomerID:",x.loc[x["CustomerID"].isna(),"importe"].sum())

### Decisiones preliminares para 1.2
| Caso | Decisión de trabajo | Justificación |
|---|---|---|
| Duplicados exactos | Eliminar una copia | Evitar doble conteo; verificar posible duplicación legítima |
| Facturas `C` | Conservar si cantidad negativa y precio positivo | Restan al día de la devolución |
| Otros prefijos no numéricos | Excluir de serie, revisar listado | No se identifican como ventas o cancelaciones normales |
| `StockCode` administrativo | Excluir solo códigos identificados | No corresponde a productos vendidos |
| `StockCode` alfanumérico legítimo | Conservar | No se debe eliminar solo por el formato |
| Precio no positivo | Excluir | No representa venta a precio positivo |
| Cantidad negativa sin `C` | Excluir y documentar | Posible ajuste no comercial |
| Sin `CustomerID` | Conservar para ventas; excluir en RFM | La venta existe aunque no se identifique cliente |

**Criterio sujeto a revisión:** códigos como POST y DOT pueden representar cobros vinculados a pedidos. La consigna pide justificar la definición de *venta de productos*. Si el profesor entiende venta neta incluyendo cargos de envío, hay que ajustar `CODIGOS_ADMIN` y recalcular. No afirmar que existe una única limpieza correcta.

## 1.3 Función de limpieza y bitácora

In [ ]:
limpio, bitacora = limpiar(df, devolver_bitacora=True)
display(bitacora)
print("Filas originales:",len(df))
print("Filas limpias:",len(limpio))
print("Filas eliminadas:",int(bitacora["filas_eliminadas"].sum()))
print("Importe bruto antes:",round(x["importe"].sum(),2))
print("Importe neto después:",round(limpio["importe"].sum(),2))
print("Verificación de filas:",len(df)==len(limpio)+bitacora["filas_eliminadas"].sum())
bitacora.to_csv("parte1_bitacora.csv",index=False,encoding="utf-8-sig")

## 1.4 Seis líneas de mayor importe absoluto y seguimiento de facturas

In [ ]:
top6 = x.assign(abs_importe=x["importe"].abs()).nlargest(6,"abs_importe")
display(top6[["InvoiceNo","InvoiceDate","StockCode","Description","Quantity","UnitPrice","importe"]])
for _, fila in top6.iterrows():
    print("\n--- Factura",fila["InvoiceNo"],"StockCode",fila["StockCode"],"---")
    misma_factura = x.loc[x["InvoiceNo"] == fila["InvoiceNo"]]
    display(misma_factura[["InvoiceNo","InvoiceDate","StockCode","Description","Quantity","UnitPrice","importe"]].head(20))
    relacionados = x.loc[(x["StockCode"]==fila["StockCode"]) &
                         (x["InvoiceDate"]>=fila["InvoiceDate"]) &
                         (x["InvoiceDate"]<=fila["InvoiceDate"]+pd.Timedelta(days=30))]
    if pd.notna(fila["CustomerID"]):
        relacionados=relacionados.loc[relacionados["CustomerID"]==fila["CustomerID"]]
    display(relacionados[["InvoiceNo","InvoiceDate","CustomerID","Quantity","UnitPrice","importe"]].sort_values("InvoiceDate").head(25))

**Análisis obligatorio 1.4:** Para cada una de las seis líneas, explicar qué representa, si existe reversión o cancelación posterior, si `limpiar()` la conserva o excluye, y si netear por día realmente compensa el movimiento. **No asumir** que una devolución posterior se compensa el mismo día: la consigna asigna cancelaciones al día en que ocurren.

In [ ]:
# Verificación: ¿sobrevivieron las seis líneas a la limpieza?
# Comparación por columnas originales (puede haber varias filas idénticas).
columnas=["InvoiceNo","StockCode","Description","Quantity","InvoiceDate","UnitPrice","CustomerID","Country"]
originales = top6[columnas].copy()
conservadas = limpio[columnas].drop_duplicates()
revision = originales.merge(conservadas.assign(conservada=True),
                             on=columnas,how="left")
revision["conservada"] = revision["conservada"].fillna(False)
display(revision[["InvoiceNo","StockCode","Quantity","UnitPrice","conservada"]])

## 1.5 Función reutilizable y salida para Parte 2

In [ ]:
# limpiar(df) está definida en limpieza.py, como exige el profesor.
assert "importe" in limpio.columns
assert limpio["InvoiceDate"].notna().all()
assert (limpio["UnitPrice"]>0).all()
assert (limpio["Quantity"]!=0).all()
assert not ((limpio["Quantity"]<0)&~limpio["es_cancelacion"]).any()
# Para la Parte 2: sumamos importes por fecha de transacción.
venta_diaria = (limpio.assign(fecha=limpio["InvoiceDate"].dt.normalize())
                .groupby("fecha",as_index=False)["importe"].sum()
                .rename(columns={"importe":"venta_neta_GBP"}))
display(venta_diaria.head())
print("Días con transacciones:",len(venta_diaria))
# No se rellena calendario aquí: esa decisión corresponde a Parte 2.

## Conclusión de la Parte 1 — resultados de la base real

Con las reglas reproducibles de `limpieza.py`, se eliminaron **5,268 duplicados exactos**, **3 registros de factura con prefijo no reconocido**, **2,907 líneas administrativas** y **2,495 líneas con precio no positivo**. En total se excluyeron **10,673 filas** y permanecieron **531,236**. La venta neta resultante fue **£9,772,003.97**. La bitácora `parte1_bitacora.csv` registra el orden, el importe excluido y la justificación.

**Criterio metodológico:** se excluyeron cargos administrativos y de envío para definir venta neta de productos; las cancelaciones `C` válidas restan en su fecha. Este criterio no es único: documentarlo y revisar los seis movimientos extremos del inciso 1.4 antes de concluir que todos los casos anómalos están resueltos. El archivo `Online Retail.xlsx` se necesita para reproducir los resultados.

## Parte 2 — Calendario (2.1 a 2.4)
El calendario completo conserva días cerrados como `NaN`; el último día se excluye por incompleto. Las fechas de feriados deben verificarse con el calendario oficial británico antes de interpretar los huecos.

In [ ]:
serie = limpio.assign(fecha=limpio.InvoiceDate.dt.normalize()).groupby('fecha')['importe'].sum().sort_index()
cal = pd.date_range(serie.index.min(), serie.index.max(), freq='D')
calendario = serie.reindex(cal).rename('venta_neta')
calendario.index.name='fecha'
print('Días calendario:',len(calendario),'Días con transacciones:',serie.size)
faltantes = calendario[calendario.isna()].index
faltan = pd.DataFrame({'fecha':faltantes,'dia_semana':faltantes.day_name()})
display(faltan)
print('Días faltantes por día de semana:')
display(faltan.groupby('dia_semana').size().sort_values(ascending=False))
print('Último día:', calendario.tail(1))
calendario = calendario.loc[:'2011-12-08']
# Días sin transacciones son faltantes para modelar operación; ceros para ventas calendario totales.
entreno = calendario.loc[:'2011-09-30'].dropna()
prueba = calendario.loc['2011-10-03':'2011-12-08'].dropna()
print('Entrenamiento (abiertos):',len(entreno),'Prueba (abiertos):',len(prueba))

**2.2 — Interpretación pendiente de contraste externo:** investigar feriados británicos de 2011, incluido el feriado extraordinario de la boda real (29 de abril), y los feriados móviles de Semana Santa. Verificar que los días sin transacciones correspondan a cierres reales y no a problemas de registro.

**2.3:** Para ingresos contables del calendario, un cierre puede ser venta cero; para estimar ventas en días abiertos, el cierre es observación no disponible. **2.4:** Se elimina el 9 de diciembre por día parcial.

## Parte 3 — Tendencia, estacionalidad y autocorrelación (3.1 a 3.3)

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import chi2
# 3.1: media móvil centrada 7 días; SOLO aquí rellenar días cerrados con cero
ma7 = calendario.fillna(0).rolling(7,center=True,min_periods=7).mean()
ax=calendario.fillna(0).plot(figsize=(13,4),alpha=.4,label='Ventas por día calendario')
ma7.plot(ax=ax,label='Media móvil centrada 7 días',linewidth=2)
ax.set(title='Ventas diarias y tendencia',ylabel='GBP (£)');ax.legend();plt.show()
# 3.2: índices sobre entrenamiento, días abiertos
est = pd.DataFrame({'venta':entreno,'ma7':ma7.reindex(entreno.index)})
est['ratio']=est['venta']/est['ma7'].replace(0,np.nan)
est['dia']=est.index.day_name()
indices=est.groupby('dia')['ratio'].agg(['mean','count']).rename(columns={'mean':'indice'})
display(indices)
indices['indice'].plot.bar(title='Índice estacional por día abierto');plt.show()

In [ ]:
# 3.3 ACF manual y Ljung-Box de días abiertos (no días calendario)
def acf_prop(y,k):
    y=np.asarray(y,dtype=float); z=y-y.mean();return np.dot(z[k:],z[:-k])/np.dot(z,z)
def ljung_prop(y,m,descontar=0):
    n=len(y); q=n*(n+2)*sum(acf_prop(y,k)**2/(n-k) for k in range(1,m+1))
    return {'Q':q,'gl':m-descontar,'p':chi2.sf(q,m-descontar)}
y=entreno.to_numpy(); rezagos=np.arange(1,22)
r=np.array([acf_prop(y,k) for k in rezagos]); banda=1.96/np.sqrt(len(y))
plt.figure(figsize=(11,3));plt.stem(rezagos,r);plt.axhline(banda,ls='--');plt.axhline(-banda,ls='--');plt.title('ACF manual: entrenamiento abierto');plt.show()
print('Fuera de bandas:',sum(abs(r)>banda),'Pico:',int(rezagos[np.argmax(abs(r))]))
for m in [6,12]:print('Ljung-Box',m,ljung_prop(y,m))

## Parte 4 — Modelos AR(1) y SARIMA (4.1 a 4.5)

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
# 4.1: OLS de venta sobre indicadores de día abierto, sin intercepto
X=pd.get_dummies(entreno.index.dayofweek,prefix='dia',dtype=float)
X.index=entreno.index
mod_dias=sm.OLS(entreno.to_numpy(),X.to_numpy()).fit()
e=mod_dias.resid
print(mod_dias.summary())
plt.figure(figsize=(11,3));plt.stem(range(1,22),[acf_prop(e,k) for k in range(1,22)]);plt.title('ACF de residuos del modelo semanal');plt.show()
# 4.2: AR(1) sobre residuos
mod_ar=sm.OLS(e[1:],sm.add_constant(e[:-1])).fit()
u=mod_ar.resid
print('phi estimado:',mod_ar.params[1])
plt.figure(figsize=(11,3));plt.stem(range(1,22),[acf_prop(u,k) for k in range(1,22)]);plt.title('ACF tras AR(1)');plt.show()
for m in (6,12):print('AR(1) Q',m,ljung_prop(u,m,descontar=1))

In [ ]:
# 4.3: modelos SARIMA de la guía; log requiere ventas positivas.
if (entreno<=0).any():
    print('ATENCIÓN: hay ventas diarias no positivas; investigar fechas antes de aplicar log.')
    display(entreno[entreno<=0])
    raise ValueError('El logaritmo de ventas no positivas no está definido; documentar tratamiento.')
logy=np.log(entreno.to_numpy())
def ajustar_sarima(valores, orden, estacional, tendencia):
    m=SARIMAX(np.asarray(valores),order=orden,seasonal_order=estacional,
              trend=tendencia,enforce_stationarity=False,enforce_invertibility=False)
    ajuste=m.fit(disp=False,maxiter=500)
    print('Convergencia:',ajuste.mle_retvals.get('converged'),'AIC:',ajuste.aic)
    return ajuste
s1=ajustar_sarima(logy,(1,0,0),(1,0,0,6),'c')
s2=ajustar_sarima(logy,(0,1,1),(0,1,1,6),'n')
for nombre,aj,res0 in [('S1',s1,0),('S2',s2,7)]:
    residuos=np.asarray(aj.resid)[res0:]
    for m in (6,12): print(nombre,m,ljung_prop(residuos,m,descontar=2))

**4.3–4.5 Interpretación:** `s=6` representa seis días operativos por semana; feriados rompen la regularidad. El log estabiliza escalas pero exige valores positivos. S1 estacionario tiende a su media de largo plazo; S2 diferenciado mantiene efectos persistentes. Verificar residuos, convergencia y gráficos antes de afirmar cuál es mejor.

## Parte 5 — Pronóstico y validación sin fuga (5.1 a 5.5)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold
# B1, B2, M1 y M2. Los modelos semanales predicen con información previa al origen.
def dia_abierto_previo(hist, fecha, semanas=1):
    objetivo=pd.Timestamp(fecha)-pd.Timedelta(days=7*semanas)
    anteriores=hist[(hist.index<=objetivo)&(hist.index.dayofweek==fecha.dayofweek)]
    return float(anteriores.iloc[-1]) if len(anteriores) else np.nan

def matriz_m1(hist):
    t=(hist.index-pd.Timestamp('2010-12-01')).days.astype(float)
    d=pd.get_dummies(hist.index.dayofweek,dtype=float).reindex(columns=range(7),fill_value=0)
    return np.column_stack([np.asarray(t),d.to_numpy()])

def matriz_m2(hist):
    a=[]
    for fecha in hist.index:
        antes=hist.loc[hist.index<fecha]
        a.append([dia_abierto_previo(antes,fecha,1),dia_abierto_previo(antes,fecha,2)])
    d=pd.get_dummies(hist.index.dayofweek,dtype=float).reindex(columns=range(7),fill_value=0)
    return np.column_stack([d.to_numpy(),np.array(a)])

def pred_basicos(hist, fechas, modelo):
    if modelo=='B1':return np.repeat(hist.mean(),len(fechas))
    if modelo=='B2':return np.array([dia_abierto_previo(hist,f,1) for f in fechas])
    if modelo=='M1':
        ajuste=sm.OLS(hist.to_numpy(),matriz_m1(hist)).fit()
        fake=pd.Series(np.zeros(len(fechas)),index=pd.DatetimeIndex(fechas))
        return ajuste.predict(matriz_m1(fake))
    if modelo=='M2':
        xm=matriz_m2(hist); ok=np.isfinite(xm).all(axis=1)
        ajuste=sm.OLS(hist.to_numpy()[ok],xm[ok]).fit()
        # Para pronosticar desde un domingo, solo usar rezagos ya observados.
        a=[]
        for f in fechas:
            d=np.eye(7)[f.dayofweek]
            a.append(np.r_[d,dia_abierto_previo(hist,f,1),dia_abierto_previo(hist,f,2)])
        return ajuste.predict(np.array(a))
    raise ValueError(modelo)

def pred_sarima(hist, fechas, modelo):
    if (hist<=0).any():return np.full(len(fechas),np.nan)
    orden,est,tend= ((1,0,0),(1,0,0,6),'c') if modelo=='S1' else ((0,1,1),(0,1,1,6),'n')
    fit=SARIMAX(np.log(hist.to_numpy()),order=orden,seasonal_order=est,trend=tend,enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=500)
    if not fit.mle_retvals.get('converged'):return np.full(len(fechas),np.nan)
    # Posición entre los días abiertos posteriores al origen.
    # Para horizontes con feriados, contar solo fechas observadas en el calendario.
    horizonte=[len(calendario.loc[hist.index.max()+pd.Timedelta(days=1):f].dropna()) for f in fechas]
    valores=np.exp(np.asarray(fit.forecast(steps=max(horizonte))))
    return np.array([valores[h-1] for h in horizonte])

def origenes_domingos(inicio,fin):
    return pd.date_range(inicio,fin,freq='W-SUN')

def validar_origenes(inicio,fin,modelos=('B1','B2','M1','M2','S1','S2'), fin_evaluacion=None):
    fin_evaluacion = pd.Timestamp(fin_evaluacion or fin)
    filas=[]; predicciones=[]
    for origen in origenes_domingos(inicio,fin):
        hist=calendario.loc[:origen].dropna()
        objetivo=calendario.loc[origen+pd.Timedelta(days=1):min(origen+pd.Timedelta(days=7),fin_evaluacion)].dropna()
        if len(objetivo)==0:continue
        for modelo in modelos:
            try:
                pred=(pred_basicos(hist,objetivo.index,modelo) if modelo in ('B1','B2','M1','M2')
                      else pred_sarima(hist,objetivo.index,modelo))
                for fecha,real,predicho in zip(objetivo.index,objetivo.to_numpy(),pred):
                    predicciones.append(dict(modelo=modelo,origen=origen,fecha=fecha,real=real,predicho=predicho))
            except Exception as exc:
                print('Revisar',modelo,origen.date(),str(exc)[:120])
    tabla=pd.DataFrame(predicciones)
    if tabla.empty:return pd.DataFrame(),tabla
    tabla=tabla.dropna(subset=['predicho'])
    if len(tabla):
        display(tabla.groupby('modelo').agg(pronosticos=('predicho','size'), origenes=('origen','nunique'))) 
    resumen=tabla.groupby('modelo').apply(lambda z:pd.Series({'MAE':mean_absolute_error(z.real,z.predicho),
         'RMSE':np.sqrt(mean_squared_error(z.real,z.predicho)),'n':len(z)}),include_groups=False)
    return resumen,tabla

### 5.2(b) y (c) — Origen móvil semanal
**Advertencia de ejecución:** reajustar SARIMA cada domingo puede tardar varios minutos. Ejecutar primero B1–M2 y luego los seis modelos. El periodo de entrenamiento usa orígenes desde mayo hasta septiembre; la prueba inicia con origen 2 de octubre y termina el 8 de diciembre.

In [ ]:
# Evaluación completa de B1, B2, M1, M2, S1 y S2; puede tardar varios minutos.
modelos=('B1','B2','M1','M2','S1','S2')
val_ent, pred_ent=validar_origenes('2011-05-01','2011-09-25',modelos,fin_evaluacion='2011-09-30')
val_test,pred_test=validar_origenes('2011-10-02','2011-12-04',modelos,fin_evaluacion='2011-12-08')
print('Origen móvil entrenamiento:');display(val_ent)
print('Origen móvil prueba:');display(val_test)
# No aceptar tablas incompletas como comparables
print('Días previstos por modelo (entrenamiento y prueba):')
display(pd.concat({'entrenamiento':val_ent['n'],'prueba':val_test['n']},axis=1))


### 5.2(a) — Validación aleatoria (solo B1, M1 y M2)
Para M2, los rezagos se calculan de la serie cronológica **antes** de separar folds; es un diagnóstico retrospectivo y NO una validación temporal válida. No usarlo para elegir el modelo final.

In [ ]:
from sklearn.model_selection import KFold
import warnings
X1=matriz_m1(entreno); X2=matriz_m2(entreno)
comparaciones=[]
for nombre,X in [('B1',np.ones((len(entreno),1))),('M1',X1),('M2',X2)]:
    ok=np.isfinite(X).all(axis=1); xx=X[ok]; yy=entreno.to_numpy()[ok]
    for semilla in range(10):
        cv=KFold(n_splits=5,shuffle=True,random_state=semilla)
        preds=np.full(len(yy),np.nan)
        for tr,te in cv.split(xx):
            if nombre=='B1':preds[te]=yy[tr].mean()
            else:preds[te]=sm.OLS(yy[tr],xx[tr]).fit().predict(xx[te])
        comparaciones.append({'modelo':nombre,'semilla':semilla,
          'MAE':mean_absolute_error(yy,preds),'RMSE':np.sqrt(mean_squared_error(yy,preds))})
cv_tab=pd.DataFrame(comparaciones).groupby('modelo')[['MAE','RMSE']].agg(['mean','std'])
display(cv_tab)

### 5.3–5.5 — Interpretación y prueba de fuga
Comparar MAE y RMSE por día pronosticado, no promediar por semana. La media móvil centrada usa información futura y **no es un predictor legal**. Para demostrar la trampa de 5.4 hay que medir explícitamente M2 con la media móvil precalculada en validación aleatoria y origen móvil, y explicar por qué no puede evaluarse honestamente en prueba. Un año de datos no permite estimar con fiabilidad la estacionalidad anual.

### 5.3 Comparación de pronósticos en prueba
Se comparan exclusivamente pronósticos emitidos desde domingos anteriores al día evaluado. Se revisan convergencia y cobertura; un modelo con predicciones faltantes no es directamente comparable.

In [ ]:
if not val_test.empty:
    print('Mejor MAE entre modelos con pronósticos:',val_test['MAE'].idxmin())
    print('Mejor RMSE entre modelos con pronósticos:',val_test['RMSE'].idxmin())
    plt.figure(figsize=(14,5))
    reales=pred_test.drop_duplicates('fecha').sort_values('fecha')
    plt.plot(reales.fecha,reales.real,label='Real',color='black',linewidth=1.5)
    for mod in modelos:
        tmp=pred_test[pred_test.modelo==mod].sort_values('fecha')
        plt.plot(tmp.fecha,tmp.predicho,label=mod,alpha=.75)
    plt.title('Pronósticos semanales frente a ventas reales (prueba)')
    plt.ylabel('GBP');plt.legend(ncol=4);plt.xticks(rotation=30);plt.tight_layout();plt.show()


### 5.4 Experimento de fuga de información (predictor ilegal)
La media móvil **centrada** usa días posteriores a la fecha que se intenta pronosticar. Se añade solo como experimento para demostrar por qué el resultado es engañoso. **No** debe emplearse como modelo válido de pronóstico ni calcularse para prueba con valores futuros.

In [ ]:
# El predictor se calcula UNA SOLA VEZ sobre la serie completa de entrenamiento,
# lo cual ya introduce información futura en cada fecha del entrenamiento.
ma_fuga=entreno.reindex(calendario.index).fillna(0).rolling(7,center=True,min_periods=7).mean().reindex(entreno.index)
X2base=matriz_m2(entreno)
X2fuga=np.column_stack([X2base,ma_fuga.to_numpy()])
comparar_fuga=[]
for nombre,X in [('M2 sin fuga',X2base),('M2 con media centrada ILEGAL',X2fuga)]:
    valido=np.isfinite(X).all(axis=1)
    Xv=X[valido]; yv=entreno.to_numpy()[valido]
    for semilla in range(10):
        cv=KFold(n_splits=5,shuffle=True,random_state=semilla)
        pron=np.full(len(yv),np.nan)
        for tr,te in cv.split(Xv):
            pron[te]=sm.OLS(yv[tr],Xv[tr]).fit().predict(Xv[te])
        comparar_fuga.append({'modelo':nombre,'semilla':semilla,'MAE':mean_absolute_error(yv,pron),'RMSE':np.sqrt(mean_squared_error(yv,pron))})
resultado_fuga=pd.DataFrame(comparar_fuga).groupby('modelo')[['MAE','RMSE']].agg(['mean','std'])
display(resultado_fuga)
# Validación temporal contaminada, no es una evaluación válida.
def validar_m2_fuga():
    filas=[]
    for origen in pd.date_range('2011-05-01','2011-09-25',freq='W-SUN'):
        h=entreno.loc[:origen]; objetivos=entreno.loc[origen+pd.Timedelta(days=1):min(origen+pd.Timedelta(days=7),pd.Timestamp('2011-09-30'))]
        if len(objetivos)==0:continue
        # Ajustar solo hasta el origen, pero la columna MA ya fue precalculada con futuro.
        base=matriz_m2(h); X=np.column_stack([base,ma_fuga.reindex(h.index).to_numpy()])
        ok=np.isfinite(X).all(axis=1)
        fit=sm.OLS(h.to_numpy()[ok],X[ok]).fit()
        for fecha,real in objetivos.items():
            # Aquí se revela el dato futuro mediante la media móvil centrada.
            fila=np.r_[np.eye(7)[fecha.dayofweek],dia_abierto_previo(h,fecha,1),dia_abierto_previo(h,fecha,2),ma_fuga.loc[fecha]]
            if np.isfinite(fila).all(): filas.append((real,float(fit.predict(fila.reshape(1,-1))[0])))
    if not filas:return None
    arr=np.asarray(filas)
    return {'MAE':mean_absolute_error(arr[:,0],arr[:,1]),'RMSE':np.sqrt(mean_squared_error(arr[:,0],arr[:,1]),'n':len(arr)}
print('Validación semanal contaminada de M2:',validar_m2_fuga())
print('NO se calcula prueba 5.4(c): el predictor exige observar el futuro.')


### 5.5 Interpretación de validación
La validación aleatoria rompe el orden temporal y mezcla regímenes de demanda. La validación semanal respeta la información disponible al origen, pero puede subestimar el error si la demanda cambia de nivel o de composición en octubre–diciembre. Mejorar la validación corrige la primera causa, no el cambio de distribución. Con aproximadamente un año no se puede estimar de manera robusta un ciclo anual repetido.

## Parte 6 — RFM: recencia, frecuencia y monto (6.1 a 6.5)

In [ ]:
from sklearn.metrics import roc_auc_score
clientes=limpio.loc[limpio.CustomerID.notna()].copy()
clientes['fecha']=clientes.InvoiceDate.dt.normalize()
pre=clientes.loc[clientes.fecha<pd.Timestamp('2011-10-01')].copy()
post=clientes.loc[(clientes.fecha>=pd.Timestamp('2011-10-01'))&(clientes.fecha<=pd.Timestamp('2011-12-08'))].copy()
ventas=pre.loc[(~pre.es_cancelacion)&(pre.Quantity>0)]
ult=ventas.groupby('CustomerID')['fecha'].max()
fre=ventas.groupby('CustomerID')['InvoiceNo'].nunique()
mon=pre.groupby('CustomerID')['importe'].sum()
rfm=pd.DataFrame({'ultima_compra':ult,'F':fre,'M':mon}).dropna(subset=['ultima_compra']).fillna({'F':0,'M':0})
rfm['R']=(pd.Timestamp('2011-10-01')-rfm.ultima_compra).dt.days
print('Clientes con compras antes de corte:',len(rfm),'M<=0:',int((rfm.M<=0).sum()))
# 6.2: empates en F con orden aleatorio reproducible, según la consigna
rfm=rfm.sort_index(); rng=np.random.default_rng(2026)
rfm=rfm.iloc[rng.permutation(len(rfm))].copy()
for columna,asc in [('R',False),('F',True),('M',True)]:
    rfm['puntaje_'+columna]=pd.qcut(rfm[columna].rank(method='first',ascending=asc),5,labels=[1,2,3,4,5]).astype(int)
rfm['suma_puntajes']=rfm[['puntaje_R','puntaje_F','puntaje_M']].sum(axis=1)
rfm['compro']=(rfm.index.isin(post.loc[(post.Quantity>0)&(~post.es_cancelacion),'CustomerID'])).astype(int)
for col in ['puntaje_R','puntaje_F','puntaje_M']:
    print(col);display(rfm.groupby(col)['compro'].agg(['mean','size']))
for col,signo in [('R',-1),('F',1),('M',1),('suma_puntajes',1)]:
    if rfm.compro.nunique()==2: print('AUC',col,roc_auc_score(rfm.compro,signo*rfm[col]))

In [ ]:
# 6.4 y 6.5: denominador = venta neta post de TODOS los clientes identificados
campeones=rfm.loc[(rfm[['puntaje_R','puntaje_F','puntaje_M']]>=4).all(axis=1)].index
monto_total=post.importe.sum()
monto_campeones=post.loc[post.CustomerID.isin(campeones),'importe'].sum()
clientes_nuevos=set(post.CustomerID.dropna())-set(ventas.CustomerID.dropna())
monto_nuevos=post.loc[post.CustomerID.isin(clientes_nuevos),'importe'].sum()
print('Fracción de clientes campeones:',len(campeones)/len(rfm))
print('Fracción venta neta campeones:',monto_campeones/monto_total if monto_total else np.nan)
print('Fracción venta neta nuevos:',monto_nuevos/monto_total if monto_total else np.nan)
rfm.to_csv('parte6_rfm.csv',encoding='utf-8-sig')

**6.5 — Discusión:** Un modelo RFM entrenado sobre clientes históricos no puede anticipar directamente la demanda de clientes que todavía no existen en la base; un enfoque agregado *top-down* puede incorporar esa demanda dentro del total, mientras que el enfoque *bottom-up* necesita un modelo adicional de adquisición de clientes.

## Lista de comprobación antes de entregar
- [ ] Ejecutar notebook completo con `Online Retail.xlsx` y revisar excepciones.
- [ ] Revisar y justificar los seis casos extremos de 1.4.
- [ ] Verificar días cerrados, feriados y el último día parcial.
- [ ] Completar S1 y S2 en validación móvil; añadir análisis de 5.4.
- [ ] Interpretar las tablas y gráficas con números reales.
- [ ] Incorporar todo el análisis al reporte `.tex` y `.pdf`.
- [ ] Subir `limpieza.py`, notebook y `README.md` a GitHub.

**Este notebook es un borrador de ejecución y análisis, no una entrega completamente validada.**